# OpenEnv Inference on H100

This notebook loads a trained checkpoint or base model, builds an OpenEnv observation prompt, generates one action, and optionally steps the simulator.

In [ ]:
%pip install -q -U torch transformers

In [ ]:
import torch

from training_script import build_training_prompt, generate_action_with_model, load_model_artifacts
from server.hackathon_environment import BioExperimentEnvironment

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
MODEL_PATH = "artifacts/grpo-h100"  # or a Hugging Face repo / base model id
SCENARIO_NAME = "cardiac_disease_de"


tokenizer, model = load_model_artifacts(
    MODEL_PATH,
    trust_remote_code=True,
)

env = BioExperimentEnvironment(scenario_name=SCENARIO_NAME)
obs = env.reset(seed=42)
prompt = build_training_prompt(obs)
print(prompt[:2000])

In [ ]:
result = generate_action_with_model(
    model,
    tokenizer,
    obs,
    max_new_tokens=220,
    temperature=0.2,
    top_p=0.9,
    do_sample=True,
)

print(result["response_text"])
result["action"]

In [ ]:
if result["action"] is not None:
    next_obs = env.step(result["action"])
    print("Reward:", next_obs.reward)
    print("Done:", next_obs.done)
    print("Violations:", next_obs.rule_violations)
    if next_obs.latest_output is not None:
        print("Summary:", next_obs.latest_output.summary)
else:
    print("Model output did not parse into an ExperimentAction.")

In [ ]:
# Optional short rollout loop.
obs = env.reset(seed=7)
trajectory = []

for step_idx in range(6):
    result = generate_action_with_model(model, tokenizer, obs, max_new_tokens=220)
    action = result["action"]
    trajectory.append({
        "step": step_idx + 1,
        "response_text": result["response_text"],
        "action": action.model_dump() if action is not None else None,
    })
    if action is None:
        break
    obs = env.step(action)
    if obs.done:
        break

trajectory